# Một giải pháp kinh doanh đầy đủ

## Bây giờ chúng ta sẽ nâng dự án Day 1 lên mức tiếp theo

### THÁCH THỨC KINH DOANH:

Tạo một sản phẩm dựng Brochure (tờ giới thiệu) cho một công ty, dùng cho khách hàng tiềm năng, nhà đầu tư và ứng viên tiềm năng.

Chúng ta sẽ được cung cấp tên công ty và website chính của họ.

Xem cuối notebook này để có ví dụ ứng dụng kinh doanh thực tế.

Và nhớ: tôi luôn sẵn sàng nếu bạn gặp vấn đề hoặc có ý tưởng! Hãy liên hệ.

In [2]:
# imports (nhập thư viện)
# Nếu các dòng này thất bại, hãy kiểm tra bạn đang chạy trong môi trường đã 'activated' (kích hoạt) với (llms) trên command prompt (dòng lệnh)

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [3]:
# Khởi tạo và các constants (hằng số)

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [4]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## Bước đầu tiên: Để GPT-5-nano tìm ra những link (liên kết) nào là relevant (liên quan)

### Dùng một lời gọi gpt-5-nano để đọc các link trên webpage (trang web), rồi trả lời bằng JSON có cấu trúc.  
Nó nên quyết định link nào là relevant, và thay relative links (liên kết tương đối) như "/about" bằng "https://company.com/about".  
Chúng ta sẽ dùng "one shot prompting" (prompt một ví dụ) — cung cấp một example (ví dụ) về cách nó nên trả lời ngay trong prompt.

Đây là một use case (tình huống sử dụng) xuất sắc cho LLM, vì nó đòi hỏi hiểu biết tinh tế. Hãy tưởng tượng phải code việc này mà không có LLM, bằng cách parse (phân tích cú pháp) và phân tích webpage — sẽ rất khó!

Sidenote (ghi chú thêm): có một kỹ thuật nâng cao hơn gọi là "Structured Outputs" (đầu ra có cấu trúc), trong đó chúng ta yêu cầu model trả lời theo một spec (đặc tả). Chúng ta sẽ học kỹ thuật này ở Week 8 trong dự án Agentic AI tự chủ.

In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [ ]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
print(get_links_user_prompt("https://edwarddonner.com"))

In [ ]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [ ]:
select_relevant_links("https://huggingface.co")

## Bước thứ hai: làm brochure (tờ giới thiệu)!

Ghép tất cả chi tiết vào một prompt khác gửi tới GPT-5-nano

In [ ]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [ ]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Hoặc bỏ comment các dòng dưới để có brochure hài hước hơn — điều này cho thấy việc đưa 'tone' (giọng điệu) vào dễ đến mức nào:

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate (cắt ngắn) nếu hơn 5.000 characters (ký tự)
    return user_prompt

In [ ]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

In [ ]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [ ]:
create_brochure("HuggingFace", "https://huggingface.co")

## Cuối cùng — một cải tiến nhỏ

Với một điều chỉnh nhỏ, chúng ta có thể đổi để kết quả stream (luồng chảy) về từ OpenAI,
với hiệu ứng typewriter (máy đánh chữ) quen thuộc

In [ ]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

In [ ]:
# Hãy thử đổi system prompt (prompt hệ thống) sang phiên bản hài hước khi làm Brochure cho Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Ứng dụng kinh doanh</h2>
            <span style="color:#181;">Trong bài tập này chúng ta mở rộng code Day 1 để thực hiện nhiều lời gọi LLM, rồi sinh ra một document (tài liệu).

Đây có lẽ là ví dụ đầu tiên về Agentic AI design patterns (mẫu thiết kế AI dạng agent), vì chúng ta kết hợp nhiều lời gọi LLM. Điều này sẽ xuất hiện nhiều hơn ở Week 2, rồi chúng ta sẽ quay lại Agentic AI quy mô lớn ở Week 8 khi xây một giải pháp Agent tự chủ hoàn chỉnh.

Sinh nội dung theo cách này là một trong những Use Cases (tình huống sử dụng) phổ biến nhất. Giống summarization (tóm tắt), có thể áp dụng cho bất kỳ lĩnh vực kinh doanh nào. Viết nội dung marketing, sinh hướng dẫn sản phẩm từ một spec (đặc tả), tạo email cá nhân hóa, và còn nhiều nữa. Hãy khám phá cách áp dụng content generation (sinh nội dung) vào công việc của bạn, và thử làm một proof-of-concept prototype (bản mẫu chứng minh ý tưởng). Xem học viên khác đã làm gì trong thư mục community-contributions — rất nhiều dự án giá trị — thật điên rồ!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Trước khi sang Week 2 (rất vui đó)</h2>
            <span style="color:#900;">Hãy xem notebook EXERCISE của week1 cho thử thách cuối tuần 1. Việc này sẽ cho bạn luyện tập thiết yếu khi làm việc với Frontier APIs (API các mô hình tiên phong), và chuẩn bị tốt cho Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Nhắc lại 3 tài nguyên hữu ích</h2>
            <span style="color:#f71;">1. Tài nguyên khóa học có <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">tại đây.</a><br/>
            2. Tôi có LinkedIn <a href="https://www.linkedin.com/in/eddonner/">tại đây</a> và rất thích kết nối với người đang học khóa này!<br/>
            3. Tôi đang thử X/Twitter tại <a href="https://x.com/edwarddonner">@edwarddonner<a> và mong mọi người chỉ cho tôi cách dùng..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Cuối cùng! Tôi có một lời nhờ đặc biệt với bạn</h2>
            <span style="color:#090;">
                Biên tập viên của tôi nói rằng việc học viên đánh giá khóa học trên Udemy tạo ra sự khác biệt RẤT LỚN — đó là một trong những cách chính để Udemy quyết định có hiện khóa này cho người khác không. Nếu bạn dành được một phút để đánh giá, tôi sẽ biết ơn vô cùng! Và dù sao đi nữa — luôn hãy liên hệ ed@edwarddonner.com nếu tôi có thể giúp bất cứ lúc nào.
            </span>
        </td>
    </tr>
</table>